# Pisciber: entrenamiento completo v3
Reproduce el protocolo para horizontes de 1, 6 y 12 horas. Sube el paquete de entrenamiento, no el ZIP de resultados.

In [ ]:
# ENTRENAMIENTO COMPLETO V3 EN COLAB (puede tardar varios minutos)
# Reproduccion: estos test ya se evaluaron localmente; no son test nuevos.
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, json, hashlib, datetime
print('Selecciona Pisciber_Entrenamiento_v3.zip')
subidos_v3=files.upload()
nombre_v3=next(n for n in subidos_v3 if n.endswith('.zip'))
raiz_v3=Path('/content')/('pisciber_entrenamiento_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
raiz_v3.mkdir()
z_v3=zipfile.ZipFile(nombre_v3)
assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z_v3.namelist())
manifiesto_v3=json.loads(z_v3.read('manifest.json'))
assert all(hashlib.sha256(z_v3.read(n)).hexdigest()==h for n,h in manifiesto_v3.items())
z_v3.extractall(raiz_v3)
z_v3.close()
print('Paquete verificado. Pruebas de historia temporal y objetivos...')
subprocess.run([sys.executable,'-m','unittest','test_mejora','-v'],cwd=raiz_v3,check=True)
print('Iniciando 27 ajustes de validacion cruzada, calibracion y prueba...')
subprocess.run([sys.executable,'-u','mejorar_predicciones.py'],cwd=raiz_v3,check=True)
print('Verificando modelos y predicciones guardadas...')
subprocess.run([sys.executable,'-u','revisar_mejora.py'],cwd=raiz_v3,check=True)
exec(compile((raiz_v3/'mostrar_resultados_v3.py').read_text(),str(raiz_v3/'mostrar_resultados_v3.py'),'exec'),{'__file__':str(raiz_v3/'mostrar_resultados_v3.py')})
files.download(str(raiz_v3/'Resultados_Pisciber_v3.zip'))
print('ENTRENAMIENTO COMPLETO TERMINADO EN COLAB')
